In [1]:
!pip install git+https://github.com/patil-suraj/vqgan-jax.git

  Cloning https://github.com/patil-suraj/vqgan-jax.git to /tmp/pip-req-build-ku3bhzqj
  Running command git clone --filter=blob:none --quiet https://github.com/patil-suraj/vqgan-jax.git /tmp/pip-req-build-ku3bhzqj
  Resolved https://github.com/patil-suraj/vqgan-jax.git to commit 10ef240f8ace869e437f3c32d14898f61512db12
  Preparing metadata (setup.py) ... done
  Created wheel for vqgan-jax: filename=vqgan_jax-0.0.1-py3-none-any.whl size=7767 sha256=350cefea7d87c29118bf4be041b08f3d148de6fe2dffff538230cf2736c817d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-rss7ru_1/wheels/15/37/2b/363deb2a1930cb5e63435aa53435925992a6914b0411b3a5c0
Successfully built vqgan-jax


In [2]:
!pip install jax jaxlib

In [3]:
import io
import os
import shutil
import requests
import time
import numpy as np
from PIL import Image
from math import nan
import math
import matplotlib.pyplot as plt
import pickle
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, ConcatDataset, DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.cuda.amp import autocast, GradScaler

import jax
import jax.numpy as jnp
import transformers
from transformers.modeling_flax_utils import FlaxPreTrainedModel
from vqgan_jax.modeling_flax_vqgan import VQModel

In [4]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
model_name = "dalle-mini/vqgan_imagenet_f16_16384"
model_vaq = VQModel.from_pretrained(model_name)

config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

flax_model.msgpack:   0%|          | 0.00/304M [00:00<?, ?B/s]

In [7]:
class Model_Z(nn.Module):
    def __init__(self):
        super(Model_Z, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=256, out_channels=2048, kernel_size=3, padding=1)
        self.batchnorm = nn.BatchNorm2d(2048)
        self.conv2 = nn.Conv2d(in_channels=2048, out_channels=256, kernel_size=3, padding=1)

        self.elu = nn.ELU()

    def forward(self, x):
        res = x
        x = self.elu(self.conv1(x))
        x = self.batchnorm(x)
        out = self.elu(self.conv2(x)) + res
        return out


In [8]:
def tensor_jax(x):
    # Check if the tensor has 3 dimensions (C, H, W)
    if x.dim() == 3:
        # Add batch dimension (N=1) to make it (1, C, H, W)
        x = x.unsqueeze(0)

    # Now permute (N, C, H, W) -> (N, H, W, C)
    x_np = x.detach().permute(0, 2, 3, 1).cpu().numpy()  # Convert to numpy
    x_jax = jnp.array(x_np)  # Convert numpy array to JAX array
    return x_jax


In [9]:
def jax_to_tensor(x):
    x_tensor = torch.tensor(np.array(x),requires_grad=True).permute(0, 3, 1, 2).to(device)  # Convert from (N, H, W, C) to (N, C, H, W)
    return x_tensor

In [10]:
# Function to save data to pickle file
def test_history(hist_test, filename='test_history.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(hist_test, f)

In [11]:
hist_test = {"Image":[], "loss_block":[], "loss_rec":[],"total_loss":[]}

In [12]:
# Define the transform
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

In [13]:
def gen_sources(model_vaq, model_z1, model_z2, model_zdf):
    criterion = nn.MSELoss()
    model_z1.eval()
    model_z2.eval()
    model_zdf.eval()
    # Specify the path to the folder containing images
    image_folder_path = '/kaggle/input/deep-ms/dataset/test/df/'
    save_folder_path = '/kaggle/working/gen/'
    os.makedirs(save_folder_path, exist_ok=True)


    # List all image files in the folder
    image_filenames = [f for f in os.listdir(image_folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]

    with torch.no_grad():
        for image_name in image_filenames:
            start_time = time.time()
            image_path = os.path.join(image_folder_path, image_name)
            img = Image.open(image_path).convert('RGB')
            df_img = transform(img)
            df_img = df_img.unsqueeze(0)  # Change shape to (1, 3, 256, 256)
            df_img = df_img.to(device)
            #convert images: tensor --> jax_array
            df_img_jax = tensor_jax(df_img)
            #calculate quantized_code(z) for all images
            z_df,_ = model_vaq.encode(df_img_jax)
            #convert quantized_code(z): jax_array --> tensor
            z_df_tensor = jax_to_tensor(z_df)
            ##----------------------------------------------------------------------
            ##----------------------model_z1-----------------------
            outputs_z1 = model_z1(z_df_tensor)
            #generate img1
            z1_rec_jax = tensor_jax(outputs_z1)
            rec_img1 = model_vaq.decode(z1_rec_jax)
            ##----------------------------------------------------------------------
            ##----------------------model_z2-----------------------
            outputs_z2 = model_z2(z_df_tensor)
            #generate img2
            z2_rec_jax = tensor_jax(outputs_z2)
            rec_img2 = model_vaq.decode(z2_rec_jax)
            ##----------------------------------------------------------------------
            ##----------------------model_zdf-----------------------
            z_rec = outputs_z1 + outputs_z2
            outputs_zdf = model_zdf(z_rec)
            lossdf = criterion(outputs_zdf, z_df_tensor)
            #calculate dfimg reconstruction loss
            zdf_rec_jax = tensor_jax(outputs_zdf)
            rec_df = model_vaq.decode(zdf_rec_jax)
            rec_df_tensor = jax_to_tensor(rec_df)
            dfimgloss = criterion(rec_df_tensor, df_img)
            # Convert tensor back to a PIL image and save
            rec_img1 = jax_to_tensor(rec_img1)
            rec_img1 = rec_img1.squeeze(0)
            rec_img2 = jax_to_tensor(rec_img2)
            rec_img2 = rec_img2.squeeze(0)
            rec_df = jax_to_tensor(rec_df)
            rec_df = rec_df.squeeze(0)
            rec_img1_pil = T.ToPILImage()(rec_img1)
            rec_img2_pil = T.ToPILImage()(rec_img2)
            rec_df_pil = T.ToPILImage()(rec_df)
            # Extract the base name without extension (e.g., "0_1" from "0_1.jpg")
            base_name = os.path.splitext(image_name)[0]  # "0_1"
            # Split the base name to get the two parts (e.g., "0" and "1")
            parts = base_name.split('_')
            f = parts[0]  # "0"
            s = parts[1]  # "1"
            # Save the images
            rec_img1_pil.save(os.path.join(save_folder_path, f'{base_name}_{f}.jpg'))  # F
            rec_img2_pil.save(os.path.join(save_folder_path, f'{base_name}_{s}.jpg'))  # G
            rec_df_pil.save(os.path.join(save_folder_path, f'{base_name}_rec.jpg'))  # H
            #Save history of each image
            hist_test["Image"].append(str(image_name))
            hist_test["loss_block"].append(lossdf.item())
            hist_test["loss_rec"].append(dfimgloss.item())
            hist_test["total_loss"].append(lossdf.item()+dfimgloss.item())
            test_history(hist_test)
            p_time = time.time() - start_time
            print(f"Processed:{image_name}.jpg',Time:{p_time:.2f} seconds, Loss-df-z:{lossdf.item():.4f}, Loss-df_rec:{dfimgloss.item():.4f}, Total-loss:{lossdf.item()+dfimgloss.item():4f}")


In [14]:
model_z1 = Model_Z()
model_z1 = model_z1.to(device)
model_z1.load_state_dict(torch.load("/kaggle/input/model_71/pytorch/default/1/model_z1.pth"))

model_z2 = Model_Z()
model_z2 = model_z2.to(device)
model_z2.load_state_dict(torch.load("/kaggle/input/model_71/pytorch/default/1/model_z2.pth"))

model_zdf = Model_Z()
model_zdf = model_zdf.to(device)
model_zdf.load_state_dict(torch.load("/kaggle/input/model_71/pytorch/default/1/model_zdf.pth"))


<All keys matched successfully>

In [15]:
gen_sources(model_vaq, model_z1, model_z2, model_zdf)

Processed:3_2.jpg.jpg',Time:2.49 seconds, Loss-df-z:0.1954, Loss-df_rec:0.0304, Total-loss:0.225826
Processed:11_12.jpg.jpg',Time:2.25 seconds, Loss-df-z:0.1634, Loss-df_rec:0.0303, Total-loss:0.193748
Processed:7_8.jpg.jpg',Time:2.24 seconds, Loss-df-z:0.1557, Loss-df_rec:0.0401, Total-loss:0.195816
Processed:2_3.jpg.jpg',Time:2.18 seconds, Loss-df-z:0.1477, Loss-df_rec:0.0254, Total-loss:0.173023
Processed:13_12.jpg.jpg',Time:2.20 seconds, Loss-df-z:0.1737, Loss-df_rec:0.0330, Total-loss:0.206791
Processed:15_16.jpg.jpg',Time:2.19 seconds, Loss-df-z:0.1568, Loss-df_rec:0.0211, Total-loss:0.177873
Processed:14_15.jpg.jpg',Time:2.35 seconds, Loss-df-z:0.1701, Loss-df_rec:0.0275, Total-loss:0.197616
Processed:15_14.jpg.jpg',Time:2.21 seconds, Loss-df-z:0.1583, Loss-df_rec:0.0203, Total-loss:0.178655
Processed:9_2.jpg.jpg',Time:2.20 seconds, Loss-df-z:0.1629, Loss-df_rec:0.0486, Total-loss:0.211536
Processed:12_13.jpg.jpg',Time:2.20 seconds, Loss-df-z:0.1514, Loss-df_rec:0.0209, Total-lo

In [16]:
# Define the folder path and output zip file path
folder_to_zip = '/kaggle/working/gen'
output_zip = '/kaggle/working/gen.zip'

# Zip the folder
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', folder_to_zip)
print("Folder zipped successfully.")

Folder zipped successfully.
